In [1]:
import pandas as pd
from os.path import join, isfile
from os import listdir
import numpy as np

In [3]:
datafiles = [join('.\\data', f) for f in listdir('./data') if join('./data', f).endswith(".csv")]

In [5]:
counter = 0
finished_counter = 0
sm_frames = []

for f in datafiles:
    if "_sm_" in f:
        print(f)
        if "task_break.started" in pd.read_csv(f): # participant did at least 1 full run
            df = pd.read_csv(f, converters={'PID': str})

            sm = df.loc[df["thisN"] >= 0]
            sm = sm.filter(['PID', 'date','blocks.thisRepN','runs.thisRepN','CurrentItem', 'CurrentITI',
                           'wait_for_trigger.stopped', 'semantic_map_item.started', 'semantic_mapping_run.living_nonliving.keys',
                           'semantic_mapping_run.living_nonliving.rt'])
            #sm.sort_values(by=['blocks.thisRepN','runs.thisRepN','CurrentItem'], inplace=True)
            sm_frames.append(sm)
            
            counter += 1
            
            if "thanks.started" in df:
                finished_counter += 1
                
print('\nNumber of experiments', counter)
print('Number of finished experiments', finished_counter)

.\data\Guess_1029_sm_2025-09-22_12h58.39.487.csv
.\data\Guess_1167_sm_2025-07-02_19h13.56.366.csv
.\data\Guess_1228_sm_2025-10-10_12h15.30.346.csv
.\data\Guess_1269_sm_2025-09-20_11h21.11.455.csv
.\data\Guess_1311_sm_2025-10-10_14h41.06.143.csv
.\data\Guess_1771_sm_2025-10-28_13h59.04.988.csv
.\data\Guess_2172_sm_2025-06-23_16h27.15.471.csv
.\data\Guess_2264_sm_2025-10-21_14h06.39.763.csv
.\data\Guess_2264_sm_2025-10-21_14h07.16.214.csv
.\data\Guess_2389_sm_2025-10-28_16h31.09.017.csv
.\data\Guess_2405_sm_2025-06-18_19h52.42.236.csv
.\data\Guess_2594_sm_2025-10-22_20h13.04.433.csv
.\data\Guess_3084_sm_2025-05-30_14h00.35.729.csv
.\data\Guess_3098_sm_2025-10-31_09h55.28.706.csv
.\data\Guess_3141_sm_2025-09-18_14h33.56.945.csv
.\data\Guess_4110_sm_2025-06-06_14h00.43.501.csv
.\data\Guess_4215_sm_2025-10-17_18h33.54.486.csv
.\data\Guess_4660_sm_2025-07-28_18h59.33.053.csv
.\data\Guess_4954_sm_2025-07-09_19h00.19.535.csv
.\data\Guess_5613_sm_2025-06-04_19h01.43.631.csv
.\data\Guess_5720_sm

In [7]:
sm_frames[0]

,PID,date,blocks.thisRepN,runs.thisRepN,CurrentItem,CurrentITI,wait_for_trigger.stopped,semantic_map_item.started,semantic_mapping_run.living_nonliving.keys,semantic_mapping_run.living_nonliving.rt
4,1029,2025-09-22_12h58.39.487,0.0,0.0,Soße,2.2890,296.341317,310.671981,3.0,1.799963
5,1029,2025-09-22_12h58.39.487,0.0,0.0,Hilfe,3.1542,NaN,315.838477,3.0,1.116251
6,1029,2025-09-22_12h58.39.487,0.0,0.0,Ehe,4.8079,NaN,322.654814,2.0,0.749515
7,1029,2025-09-22_12h58.39.487,0.0,0.0,Haare,2.3884,NaN,327.054431,3.0,1.038250
8,1029,2025-09-22_12h58.39.487,0.0,0.0,Safari,2.2568,NaN,331.320786,2.0,1.333776
...,...,...,...,...,...,...,...,...,...,...
639,1029,2025-09-22_12h58.39.487,1.0,3.0,Ziege,2.1456,NaN,4029.260637,2.0,0.717099
640,1029,2025-09-22_12h58.39.487,1.0,3.0,Gras,2.1907,NaN,4033.460401,2.0,0.701076
641,1029,2025-09-22_12h58.39.487,1.0,3.0,Platz,2.7837,NaN,4038.243308,3.0,0.635521
642,1029,2025-09-22_12h58.39.487,1.0,3.0,Freund,2.0171,NaN,4042.259877,2.0,0.962147


In [35]:
delim = ' '
form = '%.5f'

trial_dur = 2.0

for frame in sm_frames[:1]:
    PID = frame["PID"].values[0]
    triggers = frame["wait_for_trigger.stopped"][frame["wait_for_trigger.stopped"].notna()].values
    print(PID)
    print(triggers)
    frame.dropna(axis=0, how='any', subset=['runs.thisRepN', 'blocks.thisRepN'], inplace=True)
    frame['blocks.thisRepN'] = frame['blocks.thisRepN'].astype('int').astype('string')
    frame['runs.thisRepN'] = frame['runs.thisRepN'].astype('int').astype('string')
    runs = frame.groupby(['blocks.thisRepN', 'runs.thisRepN'])
    for block in range(2):
        block = str(block)
        for this_run in range(4):
            run_index = this_run
            if block == '1':
                run_index += 4
            
            this_run = str(this_run)
            
        #try:
            items_onset = runs.get_group((block, this_run))['semantic_map_item.started'] - triggers[run_index]
            items = runs.get_group((block, this_run))['CurrentItem']
            items.to_csv('onset_files/single_trials/sm/' + PID + '_Items_SM_run-0' + str(run_index+1) + '.csv',
                         header=False, index=False)
            items_onset.to_csv('onset_files/single_trials/sm/' + PID + '_Item_Onsets_SM_run-0' + str(run_index+1) + '.csv',
                         header=False, index=False)
            for i, item in enumerate(zip(items.values, items_onset.values)):
                item_list = items.to_list()
                onset_list = items_onset.to_list()
                print(item)
                item_onset = pd.DataFrame(data={'onset':[item[1]], 'duration':[trial_dur], 'strength':[1.0]})
                item_onset.to_csv('onset_files/single_trials/sm/'
                           + PID + '_SM_run-0' + str(run_index+1) + '_' + item[0] + '.csv',
                                  header=False, sep=delim, index=False, float_format="%.5f")
                print(item_onset)
                item_list.pop(i)
                onset_list.pop(i)

                comp_items_onset = pd.DataFrame(data={'onset':onset_list, 'duration':np.ones(len(onset_list)).T * trial_dur, 'strength':np.ones(len(onset_list)).T})
                comp_items_onset.to_csv('onset_files/single_trials/sm/'
                           + PID + '_SM_run-0' + str(run_index+1) + '_' + item[0] + '-LeftOut.csv',
                           header=False, sep=delim, index=False, float_format="%.5f")
        #except:
        #    print('missing run: ', block, this_run)

1029
[ 296.3413169  740.7326672 1212.6889057 1665.5465329 2270.3437247
 2728.7843155 3183.8754507 3637.1510451]
('Soße', 14.330664099999922)
       onset  duration  strength
0  14.330664       2.0       1.0
('Hilfe', 19.49715970000034)
      onset  duration  strength
0  19.49716       2.0       1.0
('Ehe', 26.313497400000415)
       onset  duration  strength
0  26.313497       2.0       1.0
('Haare', 30.71311439999954)
       onset  duration  strength
0  30.713114       2.0       1.0
('Safari', 34.97946950000005)
      onset  duration  strength
0  34.97947       2.0       1.0
('Lokomotive', 39.59616720000031)
       onset  duration  strength
0  39.596167       2.0       1.0
('Kerzen', 43.79657839999982)
       onset  duration  strength
0  43.796578       2.0       1.0
('Bürste', 48.1126776000001)
       onset  duration  strength
0  48.112678       2.0       1.0
('Zigarette', 54.16162879999956)
       onset  duration  strength
0  54.161629       2.0       1.0
('Zapfen', 59.9611255000000